# GRAM — Paper-Scale Training

Reproduction of *Generative Recursive Reasoning* (Baek et al., arXiv:2605.19376).

This notebook supports four tasks via a single switcher cell below:

| TASK         | N  | β     | metric                      |
| ---          | -- | ---   | ---                         |
| `nqueens`    | 8  | 0.07  | full_token / full_board     |
| `nqueens`    | 10 | 0.045 | full_token / full_board     |
| `graphcolor` | 8  | 0.5   | conflict edges (paper: 2.7) |
| `graphcolor` | 10 | 0.45  | conflict edges (paper: 3.3) |

Files this notebook expects in the same directory:
- `gram_model.py` — model (RoPE + ACT halt + EMA + LPRM all on)
- `data_nqueens.py` — N-Queens batcher
- `data_graph_coloring.py` — Graph 3-coloring batcher

Note: module is named `gram_model` (not `gram`) to avoid collision with the PyPI `gram` package.

Hardware: needs a single ~16–24 GB GPU. A4000/4090/A100 all fine.


In [ ]:
import os, time, math, json, torch
from contextlib import contextmanager
from torch.optim import AdamW
from gram_model import GRAM, GRAMConfig, EMA
from data_nqueens import NQueensBatcher
from data_graph_coloring import (
    GraphColoringBatcher, conflict_edges,
    VOCAB_SIZE as GC_VOCAB, COLOR_BASE, NUM_COLORS,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('gpu  :', torch.cuda.get_device_name(0))
    print('vram :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
torch.manual_seed(0)

@contextmanager
def _nullcontext():
    yield


In [ ]:
# ============================== TASK SWITCHER ==============================
# Pick one row and re-run the rest of the notebook.
TASK = 'nqueens'      # 'nqueens' or 'graphcolor'
N    = 8              # 8 or 10
# ===========================================================================

# Per-task β (paper Table values)
PAPER_BETA = {
    ('nqueens',    8):  0.07,
    ('nqueens',   10):  0.045,
    ('graphcolor', 8):  0.5,
    ('graphcolor',10):  0.45,
}
beta = PAPER_BETA[(TASK, N)]

if TASK == 'nqueens':
    vocab_size = 3
    seq_len    = N * N
    batcher    = NQueensBatcher(n=N)
    print(f'task             : N-Queens {N}x{N}')
    print(f'n-queens solutions: {batcher.num_solutions()}')
elif TASK == 'graphcolor':
    vocab_size = GC_VOCAB           # 6
    seq_len    = N * N + N
    print(f'caching 4096 3-colorable Erdős-Rényi graphs (n={N}, p_edge=0.4) ...')
    batcher    = GraphColoringBatcher(n=N, p_edge=0.4, cache_size=4096, seed=0)
    print(f'task             : Graph 3-Coloring N={N}')
    print(f'cached graphs    : {batcher.num_graphs()}')
else:
    raise ValueError(f'unknown TASK: {TASK}')

print(f'seq_len          : {seq_len}')
print(f'beta             : {beta}')


In [ ]:
# Paper-scale config — identical across all four tasks.
cfg = GRAMConfig(
    vocab_size = vocab_size,
    seq_len    = seq_len,
    d_model    = 512,
    n_heads    = 8,
    ffn_hidden = 512,
    n_layers   = 2,
    K          = 4,
    T          = 3,
    N_sup      = 16,
    use_attn   = True,
    use_rope   = True,
    use_halt   = True,
)
model = GRAM(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'params           : {n_params/1e6:.2f}M')
print(f'transitions/step : N_sup * T = {cfg.N_sup * cfg.T}')

ema = EMA(model, decay=0.9999)
opt = AdamW(model.parameters(), lr=1e-4, weight_decay=1.0, betas=(0.9, 0.95))

# ---- training hparams (paper-aligned)
B_per_step  = 64        # raise if VRAM allows
accum_steps = 12        # 64 * 12 = 768 effective batch (paper)
target_batch = B_per_step * accum_steps
assert target_batch == 768, f'effective batch {target_batch}, paper uses 768'

n_steps     = 30_000
warmup      = 1_000
log_every   = 200
eval_every  = 2_000
ckpt_every  = 5_000
kl_balance  = 0.8           # paper (Dreamer)
halt_weight = 0.5           # ⚠ guess
lprm_weight = 1.0           # ⚠ guess

def lr_at(step):
    if step < warmup:
        return step / max(1, warmup)
    return 1.0


In [ ]:
# Task-aware eval. For nqueens we report full_token / mask_token / full_board.
# For graphcolor we additionally report conflict edges (paper Table 2 metric).

@torch.no_grad()
def _nqueens_acc(pred, y, x, mask_token):
    mask_pos = (x == mask_token)
    correct  = (pred == y)
    full_tok   = correct.float().mean().item()
    mask_tok   = (correct & mask_pos).float().sum().item() / max(mask_pos.sum().item(), 1)
    full_board = correct.all(dim=1).float().mean().item()
    return full_tok, mask_tok, full_board

@torch.no_grad()
def _color_acc(pred_full, y_full, n):
    pred_c = pred_full[:, n*n:]
    y_c    = y_full[:, n*n:]
    correct = (pred_c == y_c)
    full_tok   = correct.float().mean().item()
    full_board = correct.all(dim=1).float().mean().item()
    return full_tok, full_board

@torch.no_grad()
def evaluate(model, batcher, batch_size, device, N_best=20, use_ema=False, ema=None):
    model.eval()
    if TASK == 'nqueens':
        x, y = batcher.sample(batch_size)
        adj = None
    else:
        x, y, _, adj = batcher.sample(batch_size)
        adj = adj.to(device)
    x, y = x.to(device), y.to(device)

    ctx = ema.swap_in(model) if use_ema else _nullcontext()
    with ctx:
        logits1 = model(x); pred1 = logits1.argmax(-1)
        logitsN, scores = model.forward_best_of_n(x, N=N_best); predN = logitsN.argmax(-1)
        if model.cfg.use_halt:
            logitsH, n_steps_h = model.forward_with_halt(x); predH = logitsH.argmax(-1)
            avg_steps = n_steps_h.float().mean().item()
        else:
            predH = pred1; avg_steps = float('nan')

        if TASK == 'nqueens':
            ft1, mt1, fb1 = _nqueens_acc(pred1, y, x, batcher.MASK)
            ftN, mtN, fbN = _nqueens_acc(predN, y, x, batcher.MASK)
            ftH, mtH, fbH = _nqueens_acc(predH, y, x, batcher.MASK)
            return dict(
                n1_full_token=ft1, n1_mask_token=mt1, n1_full_board=fb1,
                nN_full_token=ftN, nN_mask_token=mtN, nN_full_board=fbN,
                halt_full_token=ftH, halt_full_board=fbH,
                halt_avg_steps=avg_steps, score_mean=scores.mean().item(),
            )
        else:
            n = batcher.n
            ft1, fb1 = _color_acc(pred1, y, n)
            ftN, fbN = _color_acc(predN, y, n)
            ftH, fbH = _color_acc(predH, y, n)
            col1 = (pred1[:, n*n:] - COLOR_BASE).clamp(0, NUM_COLORS-1)
            colN = (predN[:, n*n:] - COLOR_BASE).clamp(0, NUM_COLORS-1)
            colH = (predH[:, n*n:] - COLOR_BASE).clamp(0, NUM_COLORS-1)
            return dict(
                n1_full_token=ft1, n1_full_board=fb1,
                n1_conflicts=conflict_edges(col1, adj).mean().item(),
                nN_full_token=ftN, nN_full_board=fbN,
                nN_conflicts=conflict_edges(colN, adj).mean().item(),
                halt_full_token=ftH, halt_full_board=fbH,
                halt_conflicts=conflict_edges(colH, adj).mean().item(),
                halt_avg_steps=avg_steps, score_mean=scores.mean().item(),
            )

def _print_eval(tag, ev):
    if TASK == 'nqueens':
        print(f'  >> {tag:3s} N=1   full_tok {ev["n1_full_token"]:.3f} mask {ev["n1_mask_token"]:.3f} board {ev["n1_full_board"]:.3f}')
        print(f'  >> {tag:3s} N=20  full_tok {ev["nN_full_token"]:.3f} mask {ev["nN_mask_token"]:.3f} board {ev["nN_full_board"]:.3f}')
        print(f'  >> {tag:3s} halt  full_tok {ev["halt_full_token"]:.3f} board {ev["halt_full_board"]:.3f} avg_steps {ev["halt_avg_steps"]:.2f}')
    else:
        print(f'  >> {tag:3s} N=1   tok {ev["n1_full_token"]:.3f} full {ev["n1_full_board"]:.3f} conflicts {ev["n1_conflicts"]:.2f}')
        print(f'  >> {tag:3s} N=20  tok {ev["nN_full_token"]:.3f} full {ev["nN_full_board"]:.3f} conflicts {ev["nN_conflicts"]:.2f}')
        print(f'  >> {tag:3s} halt  tok {ev["halt_full_token"]:.3f} full {ev["halt_full_board"]:.3f} conflicts {ev["halt_conflicts"]:.2f} avg_steps {ev["halt_avg_steps"]:.2f}')


In [ ]:
# Training loop — same shape for both tasks. graphcolor passes y_mask so
# loss/acc only counts the N color positions (adjacency is input-only).
out_prefix = f'gram_paper_{TASK}_n{N}'
log = []
t0 = time.time()

for step in range(1, n_steps + 1):
    for g in opt.param_groups:
        g['lr'] = 1e-4 * lr_at(step)

    opt.zero_grad(set_to_none=True)
    info_accum = {'loss': 0.0, 'recon': 0.0, 'kl': 0.0,
                  'lprm': 0.0, 'halt': 0.0, 'r': 0.0, 'acc': 0.0}
    for _ in range(accum_steps):
        if TASK == 'nqueens':
            x, y = batcher.sample(B_per_step); y_mask = None
        else:
            x, y, y_mask, _ = batcher.sample(B_per_step)
            y_mask = y_mask.to(device)
        x, y = x.to(device), y.to(device)
        model.train()
        loss, info = model.train_step(
            x, y, beta=beta, kl_balance=kl_balance,
            lprm_weight=lprm_weight, halt_weight=halt_weight,
            y_mask=y_mask)
        (loss / accum_steps).backward()
        for k in info_accum:
            info_accum[k] += info[k] / accum_steps

    gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    ema.update(model)

    if step == 1 or step % log_every == 0:
        msg = (f'step {step:6d} | loss {info_accum["loss"]:.4f} | recon {info_accum["recon"]:.4f} | '
               f'kl {info_accum["kl"]:.4f} | halt {info_accum["halt"]:.4f} | lprm {info_accum["lprm"]:.4f} | '
               f'r {info_accum["r"]:.3f} | acc {info_accum["acc"]:.3f} | gn {gn.item():.2f} | '
               f't {time.time()-t0:.1f}s')
        print(msg)
        log.append(dict(step=step, **info_accum, gnorm=gn.item(), t=time.time()-t0))

    if step % eval_every == 0:
        ev_raw = evaluate(model, batcher, 128, device, N_best=20, use_ema=False)
        ev_ema = evaluate(model, batcher, 128, device, N_best=20, use_ema=True, ema=ema)
        _print_eval('raw', ev_raw)
        _print_eval('EMA', ev_ema)

    if step % ckpt_every == 0:
        path = f'{out_prefix}_step{step}.pt'
        torch.save({
            'cfg': cfg.__dict__, 'model': model.state_dict(),
            'ema': ema.state_dict(), 'opt': opt.state_dict(), 'step': step,
        }, path)
        print(f'  >> ckpt saved: {path}')

torch.save({
    'cfg': cfg.__dict__, 'model': model.state_dict(),
    'ema':   ema.state_dict(), 'log':   log,
}, f'{out_prefix}_final.pt')
print(f'done — saved {out_prefix}_final.pt')


## Pass criteria

Paper-reported numbers (Table 2):
- **N-Queens 8×8**: 99.7% acc / 90.3% cov  (β=0.07)
- **N-Queens 10×10**: 89.7% acc / 57.5% cov (β=0.045)
- **Graph Coloring N=8**: 2.7 conflict edges  (β=0.5)
- **Graph Coloring N=10**: 3.3 conflict edges (β=0.45)

For **N-Queens** want:
- EMA best-of-1 `full_token > 0.99`
- EMA best-of-20 `full_board > 0.85`

For **Graph Coloring** want:
- EMA best-of-20 `conflicts ≈ 2.7` (N=8) or `≈ 3.3` (N=10)

For **both**:
- Halt `avg_steps < N_sup` when board/conflict is good — model learned to stop
- `corr(score, reward) > 0.7` — LPRM is informative
- distinct preds / 20 samples > 5 — prior has not collapsed


In [ ]:
ev = evaluate(model, batcher, 512, device, N_best=20, use_ema=True, ema=ema)
print(f'FINAL EMA eval ({TASK} N={N}, batch=512):')
for k, v in ev.items():
    print(f'  {k:20s}: {v:.4f}')


In [ ]:
# corr(score, reward) and prior diversity. corr > 0.7 means LPRM is useful.
# distinct preds / 20 > 5 means the prior hasn't collapsed.
import numpy as np
model.eval()
with ema.swap_in(model):
    if TASK == 'nqueens':
        x, y = batcher.sample(64); y_mask_eval = None
    else:
        x, y, _, _ = batcher.sample(64)
        y_mask_eval = torch.zeros_like(x, dtype=torch.bool)
        y_mask_eval[:, N*N:] = True
    x, y = x.to(device), y.to(device)
    e_x = model.encode(x).repeat_interleave(20, dim=0)
    h_T = model._run_prior_trajectory(e_x)
    scores = model.value_head(h_T).view(64, 20).cpu().numpy()
    logits = model.lm_head(model.norm_out(h_T))
    preds  = logits.argmax(-1).view(64, 20, -1)
    target = y[:, None, :].expand(-1, 20, -1)
    if y_mask_eval is None:
        rewards = (preds == target).float().mean(-1).cpu().numpy()
        diversity_preds = preds
    else:
        m = y_mask_eval[:, None, :].expand(-1, 20, -1).to(device)
        rewards = ((preds == target) & m).float().sum(-1) / m.float().sum(-1).clamp(min=1)
        rewards = rewards.cpu().numpy()
        # for diversity look only at the y-relevant slice
        diversity_preds = preds[:, :, N*N:]

    diversity = np.array([
        len(set(map(tuple, diversity_preds[i].tolist()))) for i in range(64)
    ])

from numpy import corrcoef
print('mean distinct preds / 20 samples (>1 means prior has diversity):', diversity.mean())
print('corr(score, reward) flat                                       :',
      corrcoef(scores.flatten(), rewards.flatten())[0,1])
